In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import os
import urllib.request

# Ensure the bronze directory exists
bronze_dir = "/lakehouse/default/Files/bronze"
os.makedirs(bronze_dir, exist_ok=True)

# Download NYC January 2024 hourly Air Quality data (JSON)
aq_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
    "?latitude=40.7128&longitude=-74.0060"
    "&start_date=2024-01-01&end_date=2024-01-31"
    "&hourly=pm10,pm2_5,nitrogen_dioxide"
)
aq_path = f"{bronze_dir}/nyc_air_quality_2024_01.json"

urllib.request.urlretrieve(aq_url, aq_path)
file_size_kb = round(os.path.getsize(aq_path) / 1024, 2)
print(f"Air Quality JSON downloaded successfully! Size: {file_size_kb} KB")

StatementMeta(, 21071600-39df-4c5a-b715-8e0b4cae8863, 3, Finished, Available, Finished, False)

Air Quality JSON downloaded successfully! Size: 24.23 KB


In [2]:
import json
from pyspark.sql.functions import col, to_date, to_timestamp

# Read the raw JSON file from Bronze layer
with open("/lakehouse/default/Files/bronze/nyc_air_quality_2024_01.json", "r") as f:
    raw_data = json.load(f)

hourly = raw_data["hourly"]

# Flatten parallel arrays into rows
rows = list(zip(
    hourly["time"],
    hourly["pm10"],
    hourly["pm2_5"],
    hourly["nitrogen_dioxide"]
))

# Create Spark DataFrame and enforce types
df_aq_silver = (
    spark.createDataFrame(rows, ["timestamp_str", "pm10", "pm2_5", "nitrogen_dioxide"])
    .withColumn("measurement_time", to_timestamp(col("timestamp_str")))
    .withColumn("measurement_date", to_date(col("measurement_time")))
    .drop("timestamp_str")
    .filter(col("pm2_5").isNotNull())
)

# Save as managed Delta table
df_aq_silver.write.mode("overwrite").format("delta").saveAsTable("silver_nyc_air_quality")
print(f"Silver Air Quality table created! Total hourly rows: {df_aq_silver.count()}")

StatementMeta(, 21071600-39df-4c5a-b715-8e0b4cae8863, 4, Finished, Available, Finished, False)

Silver Air Quality table created! Total hourly rows: 744


In [3]:
%%sql
CREATE OR REPLACE TABLE gold_air_quality_daily AS
SELECT
    measurement_date,
    CAST(ROUND(AVG(pm2_5), 2) AS DOUBLE) AS avg_pm2_5,
    CAST(ROUND(AVG(pm10), 2) AS DOUBLE) AS avg_pm10,
    CAST(ROUND(AVG(nitrogen_dioxide), 2) AS DOUBLE) AS avg_no2
FROM silver_nyc_air_quality
GROUP BY measurement_date
ORDER BY measurement_date;

StatementMeta(, 21071600-39df-4c5a-b715-8e0b4cae8863, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
CREATE OR REPLACE TABLE gold_taxi_air_quality_combined AS
SELECT
    t.pickup_date AS date,
    t.total_trips,
    t.total_revenue,
    t.avg_distance,
    t.avg_fare,
    a.avg_pm2_5,
    a.avg_pm10,
    a.avg_no2
FROM gold_taxi_daily_summary t
INNER JOIN gold_air_quality_daily a
    ON t.pickup_date = a.measurement_date
ORDER BY t.pickup_date;

SELECT * FROM gold_taxi_air_quality_combined LIMIT 10;

StatementMeta(, 21071600-39df-4c5a-b715-8e0b4cae8863, 7, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 10 rows and 8 fields>